# Stress Prediction v11 — Diverse ExtraTrees Ensemble

v10의 Multi-Seed 결과를 확장해, 이번에는 seed만 바꾸는 것이 아니라 **ExtraTrees의 구조도 조금씩 다르게** 구성합니다.

## 앙상블 구성

- Model A: `max_features=1`, `bootstrap=False`
- Model B: `max_features=2`, `bootstrap=False`
- Model C: `max_features="sqrt"`, `bootstrap=False`
- Model D: `max_features=1`, `bootstrap=True`
- Model E: `max_features=0.5`, `bootstrap=False`

각 모델은 서로 다른 seed를 사용합니다.

## 유지되는 기존 구조

- BP Ratio 단독
- Winsorize
- Gender 제거
- Feature copy
- Pair predictor
- 최근접 이웃 override
- 소수 둘째 자리 반올림 후보

## 탐색

- 각 ExtraTrees 구성의 단독 OOF 성능
- 구성별 예측 상관계수
- 상위 구성만 자동 선택
- Tree quantile `0.52 ~ 0.56`
- Pair weight `0.25 ~ 0.32`
- Override threshold `0.15 / 0.18 / 0.20 / None`


In [1]:
from __future__ import annotations

import os
import warnings
from itertools import combinations
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

TARGET = "stress_score"
ID_COLUMN = "ID"

RANDOM_SEED = 42
N_SPLITS = 5

N_ESTIMATORS_PER_MODEL = 700

TREE_QUANTILES = np.round(
    np.arange(0.52, 0.5601, 0.005),
    3,
).tolist()

PAIR_QUANTILE = 0.48
PAIR_WEIGHTS = np.round(
    np.arange(0.25, 0.3201, 0.01),
    2,
).tolist()

OVERRIDE_THRESHOLDS = [
    None,
    0.15,
    0.18,
    0.20,
]

ROUND_OPTIONS = [2, None]

WINSORIZE_COLUMNS = [
    "mean_working",
    "diastolic_blood_pressure",
    "cholesterol",
    "glucose",
]

FEATURE_COPY_COUNTS = {
    "mean_working": 1,
    "bmi": 4,
    "cholesterol": 2,
    "height": 2,
    "glucose": 3,
    "weight": 2,
    "cholesterol_glucose_ratio": 2,
    "bone_density": 4,
}

PAIR_COLUMNS = [
    "mean_working",
    "bmi",
    "cholesterol",
    "height",
    "glucose",
    "weight",
    "cholesterol_glucose_ratio",
    "bone_density",
]

MODEL_CONFIGS = {
    "mf1_seed42": {
        "max_features": 1,
        "bootstrap": False,
        "random_state": 42,
    },
    "mf2_seed2024": {
        "max_features": 2,
        "bootstrap": False,
        "random_state": 2024,
    },
    "sqrt_seed777": {
        "max_features": "sqrt",
        "bootstrap": False,
        "random_state": 777,
    },
    "mf1_boot_seed1024": {
        "max_features": 1,
        "bootstrap": True,
        "random_state": 1024,
    },
    "mf05_seed3407": {
        "max_features": 0.5,
        "bootstrap": False,
        "random_state": 3407,
    },
}

DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")
OUTPUT_DIR = Path("/Users/bitsaem/Desktop/stress_project/8_6")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FALLBACK = Path("/mnt/data/train(2).csv")
TEST_FALLBACK = Path("/mnt/data/test(2).csv")

print("ExtraTrees configs")
for name, config in MODEL_CONFIGS.items():
    print(name, config)


ExtraTrees configs
mf1_seed42 {'max_features': 1, 'bootstrap': False, 'random_state': 42}
mf2_seed2024 {'max_features': 2, 'bootstrap': False, 'random_state': 2024}
sqrt_seed777 {'max_features': 'sqrt', 'bootstrap': False, 'random_state': 777}
mf1_boot_seed1024 {'max_features': 1, 'bootstrap': True, 'random_state': 1024}
mf05_seed3407 {'max_features': 0.5, 'bootstrap': False, 'random_state': 3407}


## 1. 데이터 로드


In [ ]:
train_path = DATA_DIR / "train.csv"
test_path = DATA_DIR / "test.csv"
sample_path = DATA_DIR / "sample_submission.csv"

if not train_path.exists() and TRAIN_FALLBACK.exists():
    train_path = TRAIN_FALLBACK

if not test_path.exists() and TEST_FALLBACK.exists():
    test_path = TEST_FALLBACK

train_raw = pd.read_csv(train_path)
test_raw = pd.read_csv(test_path)

y = train_raw[TARGET].reset_index(drop=True)

raw_input = (
    train_raw
    .drop(columns=[TARGET, ID_COLUMN])
    .reset_index(drop=True)
)

test_input = (
    test_raw
    .drop(columns=[ID_COLUMN])
    .reset_index(drop=True)
)

numeric_cols = raw_input.select_dtypes(
    include=["number"]
).columns.tolist()

print("Train:", train_raw.shape)
print("Test :", test_raw.shape)
display(train_raw.head())


## 2. 기존 피처·Pair·Override 함수


In [3]:
def compute_iqr_bounds(
    train_frame: pd.DataFrame,
    columns: list[str],
):
    bounds = {}

    for col in columns:
        q1, q3 = train_frame[col].quantile(
            [0.25, 0.75]
        )
        iqr = q3 - q1

        bounds[col] = (
            q1 - 1.5 * iqr,
            q3 + 1.5 * iqr,
        )

    return bounds


def apply_winsorize(
    frame: pd.DataFrame,
    bounds,
):
    result = frame.copy()

    for col, (lower, upper) in bounds.items():
        result[col] = result[col].clip(
            lower,
            upper,
        )

    return result


def add_features(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    result = frame.copy()

    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]

    result["bmi"] = (
        result["weight"]
        / (height_m.pow(2) + 1e-6)
    )

    result["blood_pressure_ratio"] = (
        sbp / (dbp + 1e-6)
    )

    result["cholesterol_glucose_ratio"] = (
        result["cholesterol"]
        / (result["glucose"] + 1e-6)
    )

    result["missing_count"] = (
        result.isna().sum(axis=1)
    )

    return result


def add_feature_copies(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    result = frame.copy()

    for feature, n_copies in FEATURE_COPY_COUNTS.items():
        if feature not in result.columns:
            continue

        for i in range(1, n_copies + 1):
            result[f"{feature}__copy{i}"] = result[feature]

    return result


def make_preprocessor(
    frame: pd.DataFrame,
) -> ColumnTransformer:
    categorical = frame.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()

    numerical = frame.select_dtypes(
        include=["number"]
    ).columns.tolist()

    return ColumnTransformer([
        (
            "categorical",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="constant",
                        fill_value="missing",
                    ),
                ),
                (
                    "encoder",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=False,
                    ),
                ),
            ]),
            categorical,
        ),
        (
            "numerical",
            SimpleImputer(
                strategy="median",
                add_indicator=True,
            ),
            numerical,
        ),
    ])


def fit_extra_trees_matrix(
    train_features: pd.DataFrame,
    val_features: pd.DataFrame,
    target: pd.Series,
    config: dict,
) -> np.ndarray:
    train_input = train_features.drop(
        columns=["gender"],
        errors="ignore",
    )

    val_input = val_features.drop(
        columns=["gender"],
        errors="ignore",
    )

    train_input = add_feature_copies(train_input)
    val_input = add_feature_copies(val_input)

    preprocessor = make_preprocessor(train_input)

    train_matrix = preprocessor.fit_transform(
        train_input
    )

    val_matrix = preprocessor.transform(
        val_input
    )

    model = ExtraTreesRegressor(
        n_estimators=N_ESTIMATORS_PER_MODEL,
        min_samples_leaf=1,
        max_features=config["max_features"],
        bootstrap=config["bootstrap"],
        criterion="squared_error",
        random_state=config["random_state"],
        n_jobs=-1,
    )

    model.fit(
        train_matrix,
        target,
    )

    return np.column_stack([
        tree.predict(val_matrix)
        for tree in model.estimators_
    ])


def empirical_rank_transform(
    train_values,
    test_values,
):
    median = np.nanmedian(train_values)

    train_values = np.nan_to_num(
        train_values,
        nan=median,
    )

    test_values = np.nan_to_num(
        test_values,
        nan=median,
    )

    sorted_train = np.sort(train_values)

    train_rank = (
        np.searchsorted(
            sorted_train,
            train_values,
            side="right",
        )
        / len(train_values)
    )

    test_rank = (
        np.searchsorted(
            sorted_train,
            test_values,
            side="right",
        )
        / len(train_values)
    )

    return (
        train_rank.astype(np.float32),
        test_rank.astype(np.float32),
    )


def pair_prediction(
    train_features: pd.DataFrame,
    val_features: pd.DataFrame,
    target: pd.Series,
) -> np.ndarray:
    train_rank = {}
    val_rank = {}

    for column in PAIR_COLUMNS:
        train_rank[column], val_rank[column] = (
            empirical_rank_transform(
                train_features[column]
                .to_numpy(dtype=float),
                val_features[column]
                .to_numpy(dtype=float),
            )
        )

    target_values = target.to_numpy(dtype=float)
    neighbor_targets = []

    for first, second in combinations(
        PAIR_COLUMNS,
        2,
    ):
        distance = (
            np.abs(
                val_rank[first][:, None]
                - train_rank[first][None, :]
            )
            + np.abs(
                val_rank[second][:, None]
                - train_rank[second][None, :]
            )
        )

        nearest_idx = np.argmin(
            distance,
            axis=1,
        )

        neighbor_targets.append(
            target_values[nearest_idx]
        )

    return np.quantile(
        np.column_stack(neighbor_targets),
        PAIR_QUANTILE,
        axis=1,
    )


def nearest_neighbor_lookup(
    train_raw_frame: pd.DataFrame,
    val_raw_frame: pd.DataFrame,
    target: pd.Series,
):
    train_numeric = (
        train_raw_frame[numeric_cols]
        .copy()
    )

    val_numeric = (
        val_raw_frame[numeric_cols]
        .copy()
    )

    medians = train_numeric.median()

    train_numeric = train_numeric.fillna(
        medians
    )

    val_numeric = val_numeric.fillna(
        medians
    )

    scaler = StandardScaler()

    train_scaled = scaler.fit_transform(
        train_numeric
    )

    val_scaled = scaler.transform(
        val_numeric
    )

    tree = cKDTree(train_scaled)

    nearest_dist, nearest_idx = tree.query(
        val_scaled,
        k=1,
    )

    nearest_target = (
        target.to_numpy(dtype=float)[nearest_idx]
    )

    return nearest_dist, nearest_target


def apply_postprocess(
    prediction: np.ndarray,
    round_decimals: int | None,
    override_threshold: float | None,
    nearest_dist: np.ndarray,
    nearest_target: np.ndarray,
) -> np.ndarray:
    result = np.clip(
        prediction.copy(),
        0.0,
        1.0,
    )

    if round_decimals is not None:
        result = np.round(
            result,
            round_decimals,
        )

    if override_threshold is not None:
        mask = (
            nearest_dist
            < override_threshold
        )

        replacement = nearest_target[
            mask
        ].copy()

        if round_decimals is not None:
            replacement = np.round(
                replacement,
                round_decimals,
            )

        result[mask] = np.clip(
            replacement,
            0.0,
            1.0,
        )

    return result


## 3. Fold별 다양한 ExtraTrees 캐시 생성


In [4]:
cv = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_SEED,
)

fold_cache = []

for fold, (train_idx, val_idx) in enumerate(
    cv.split(raw_input),
    start=1,
):
    train_raw_fold = (
        raw_input.iloc[train_idx]
        .reset_index(drop=True)
    )

    val_raw_fold = (
        raw_input.iloc[val_idx]
        .reset_index(drop=True)
    )

    y_train = (
        y.iloc[train_idx]
        .reset_index(drop=True)
    )

    bounds = compute_iqr_bounds(
        train_raw_fold,
        WINSORIZE_COLUMNS,
    )

    train_w = apply_winsorize(
        train_raw_fold,
        bounds,
    )

    val_w = apply_winsorize(
        val_raw_fold,
        bounds,
    )

    train_feat = add_features(train_w)
    val_feat = add_features(val_w)

    model_matrices = {}

    for model_name, config in MODEL_CONFIGS.items():
        model_matrices[model_name] = (
            fit_extra_trees_matrix(
                train_feat,
                val_feat,
                y_train,
                config,
            )
        )

        print(
            f"Fold {fold}: {model_name} 완료"
        )

    pair_pred = pair_prediction(
        train_feat,
        val_feat,
        y_train,
    )

    nearest_dist, nearest_target = (
        nearest_neighbor_lookup(
            train_raw_fold,
            val_raw_fold,
            y_train,
        )
    )

    fold_cache.append({
        "val_idx": val_idx,
        "model_matrices": model_matrices,
        "pair_pred": pair_pred,
        "nearest_dist": nearest_dist,
        "nearest_target": nearest_target,
    })

    print(
        f"Fold {fold}/{N_SPLITS} 캐시 완료"
    )


Fold 1: mf1_seed42 완료
Fold 1: mf2_seed2024 완료
Fold 1: sqrt_seed777 완료
Fold 1: mf1_boot_seed1024 완료
Fold 1: mf05_seed3407 완료
Fold 1/5 캐시 완료
Fold 2: mf1_seed42 완료
Fold 2: mf2_seed2024 완료
Fold 2: sqrt_seed777 완료
Fold 2: mf1_boot_seed1024 완료
Fold 2: mf05_seed3407 완료
Fold 2/5 캐시 완료
Fold 3: mf1_seed42 완료
Fold 3: mf2_seed2024 완료
Fold 3: sqrt_seed777 완료
Fold 3: mf1_boot_seed1024 완료
Fold 3: mf05_seed3407 완료
Fold 3/5 캐시 완료
Fold 4: mf1_seed42 완료
Fold 4: mf2_seed2024 완료
Fold 4: sqrt_seed777 완료
Fold 4: mf1_boot_seed1024 완료
Fold 4: mf05_seed3407 완료
Fold 4/5 캐시 완료
Fold 5: mf1_seed42 완료
Fold 5: mf2_seed2024 완료
Fold 5: sqrt_seed777 완료
Fold 5: mf1_boot_seed1024 완료
Fold 5: mf05_seed3407 완료
Fold 5/5 캐시 완료


## 4. 개별 ExtraTrees 구성 성능 비교


In [5]:
individual_oof = {
    model_name: np.zeros(
        len(y),
        dtype=float,
    )
    for model_name in MODEL_CONFIGS
}

for cache in fold_cache:
    for model_name in MODEL_CONFIGS:
        matrix = cache["model_matrices"][
            model_name
        ]

        pred = np.quantile(
            matrix,
            0.54,
            axis=1,
        )

        individual_oof[model_name][
            cache["val_idx"]
        ] = np.round(
            np.clip(pred, 0.0, 1.0),
            2,
        )

individual_rows = []

for model_name, pred in individual_oof.items():
    individual_rows.append({
        "model": model_name,
        "cv_mae": mean_absolute_error(
            y,
            pred,
        ),
        "pred_std": float(
            pred.std()
        ),
    })

individual_df = (
    pd.DataFrame(individual_rows)
    .sort_values("cv_mae")
    .reset_index(drop=True)
)

print("개별 ExtraTrees 결과")
display(individual_df)

print("OOF 예측 상관계수")
display(
    pd.DataFrame(individual_oof)
    .corr()
    .round(4)
)


개별 ExtraTrees 결과


,model,cv_mae,pred_std
0,mf1_seed42,0.148653,0.192100
1,mf2_seed2024,0.148830,0.194041
2,mf1_boot_seed1024,0.151173,0.184687
3,sqrt_seed777,0.153980,0.195292
4,mf05_seed3407,0.167370,0.196267


OOF 예측 상관계수


,mf1_seed42,mf2_seed2024,sqrt_seed777,mf1_boot_seed1024,mf05_seed3407
mf1_seed42,1.0000,0.9909,0.9580,0.9864,0.8706
mf2_seed2024,0.9909,1.0000,0.9679,0.9834,0.8837
sqrt_seed777,0.9580,0.9679,1.0000,0.9586,0.9430
mf1_boot_seed1024,0.9864,0.9834,0.9586,1.0000,0.8773
mf05_seed3407,0.8706,0.8837,0.9430,0.8773,1.0000


## 5. 앙상블에 사용할 ExtraTrees 자동 선택

개별 성능이 지나치게 나쁜 모델은 제외합니다.

- 최고 개별 MAE보다 `0.010` 이상 나쁜 모델 제외
- 최소 2개, 최대 4개 모델 사용


In [6]:
best_individual_mae = float(
    individual_df.iloc[0]["cv_mae"]
)

selected_models = (
    individual_df.loc[
        individual_df["cv_mae"]
        <= best_individual_mae + 0.010,
        "model",
    ]
    .head(4)
    .tolist()
)

if len(selected_models) < 2:
    selected_models = (
        individual_df["model"]
        .head(2)
        .tolist()
    )

print("선택된 ExtraTrees 구성:", selected_models)


선택된 ExtraTrees 구성: ['mf1_seed42', 'mf2_seed2024', 'mf1_boot_seed1024', 'sqrt_seed777']


## 6. 선택된 ExtraTrees 앙상블 미세 탐색


In [7]:
search_rows = []

for tree_q in TREE_QUANTILES:
    for pair_weight in PAIR_WEIGHTS:
        for decimals in ROUND_OPTIONS:
            for threshold in OVERRIDE_THRESHOLDS:
                oof_pred = np.zeros(
                    len(y),
                    dtype=float,
                )

                for cache in fold_cache:
                    selected_matrices = [
                        cache["model_matrices"][
                            model_name
                        ]
                        for model_name in selected_models
                    ]

                    combined_matrix = np.concatenate(
                        selected_matrices,
                        axis=1,
                    )

                    tree_pred = np.quantile(
                        combined_matrix,
                        tree_q,
                        axis=1,
                    )

                    blended = (
                        (1.0 - pair_weight)
                        * tree_pred
                        + pair_weight
                        * cache["pair_pred"]
                    )

                    final_pred = apply_postprocess(
                        blended,
                        decimals,
                        threshold,
                        cache["nearest_dist"],
                        cache["nearest_target"],
                    )

                    oof_pred[
                        cache["val_idx"]
                    ] = final_pred

                mae = mean_absolute_error(
                    y,
                    oof_pred,
                )

                search_rows.append({
                    "cv_mae": mae,
                    "tree_quantile": tree_q,
                    "pair_weight": pair_weight,
                    "round_decimals": decimals,
                    "override_threshold": threshold,
                    "selected_models": ",".join(
                        selected_models
                    ),
                    "model_count": len(
                        selected_models
                    ),
                })

search_df = (
    pd.DataFrame(search_rows)
    .sort_values("cv_mae")
    .reset_index(drop=True)
)

display(search_df.head(40))

print(
    "최적 Diverse ExtraTrees OOF MAE:",
    float(search_df.iloc[0]["cv_mae"]),
)


,cv_mae,tree_quantile,pair_weight,round_decimals,override_threshold,selected_models,model_count
0,0.147803,0.540,0.28,2.0,0.20,"mf1_seed42,mf2_seed2024,mf1_boot_seed1024,sqrt...",4
1,0.147803,0.540,0.28,2.0,0.18,"mf1_seed42,mf2_seed2024,mf1_boot_seed1024,sqrt...",4
2,0.147803,0.540,0.28,2.0,0.15,"mf1_seed42,mf2_seed2024,mf1_boot_seed1024,sqrt...",4
3,0.147827,0.540,0.29,2.0,0.20,"mf1_seed42,mf2_seed2024,mf1_boot_seed1024,sqrt...",4
4,0.147827,0.540,0.29,2.0,0.18,"mf1_seed42,mf2_seed2024,mf1_boot_seed1024,sqrt...",4
5,0.147827,0.540,0.29,2.0,0.15,"mf1_seed42,mf2_seed2024,mf1_boot_seed1024,sqrt...",4
6,0.147830,0.540,0.27,2.0,0.20,"mf1_seed42,mf2_seed2024,mf1_boot_seed1024,sqrt...",4
7,0.147830,0.540,0.27,2.0,0.18,"mf1_seed42,mf2_seed2024,mf1_boot_seed1024,sqrt...",4
8,0.147830,0.540,0.27,2.0,0.15,"mf1_seed42,mf2_seed2024,mf1_boot_seed1024,sqrt...",4
9,0.147863,0.540,0.25,2.0,0.15,"mf1_seed42,mf2_seed2024,mf1_boot_seed1024,sqrt...",4


최적 Diverse ExtraTrees OOF MAE: 0.14780333333333331


## 7. 제출 후보 선택


In [8]:
def row_to_config(
    row: pd.Series,
) -> dict:
    return {
        "tree_quantile": float(
            row["tree_quantile"]
        ),
        "pair_weight": float(
            row["pair_weight"]
        ),
        "round_decimals": (
            None
            if pd.isna(
                row["round_decimals"]
            )
            else int(
                row["round_decimals"]
            )
        ),
        "override_threshold": (
            None
            if pd.isna(
                row["override_threshold"]
            )
            else float(
                row["override_threshold"]
            )
        ),
        "cv_mae": float(
            row["cv_mae"]
        ),
    }


candidate_rows = [
    ("cv_best", search_df.iloc[0]),
]

safe_rows = search_df[
    (search_df["tree_quantile"].between(
        0.535,
        0.55,
    ))
    & (search_df["pair_weight"].between(
        0.28,
        0.30,
    ))
    & (search_df["round_decimals"] == 2)
    & (
        search_df["override_threshold"]
        .isin([0.18, 0.20])
    )
]

if len(safe_rows):
    candidate_rows.append(
        ("safe", safe_rows.iloc[0])
    )

tree_heavy_rows = search_df[
    search_df["pair_weight"]
    == min(PAIR_WEIGHTS)
]

if len(tree_heavy_rows):
    candidate_rows.append(
        (
            "tree_heavy",
            tree_heavy_rows.iloc[0],
        )
    )

pair_heavy_rows = search_df[
    search_df["pair_weight"]
    == max(PAIR_WEIGHTS)
]

if len(pair_heavy_rows):
    candidate_rows.append(
        (
            "pair_heavy",
            pair_heavy_rows.iloc[0],
        )
    )

no_override_rows = search_df[
    search_df["override_threshold"].isna()
]

if len(no_override_rows):
    candidate_rows.append(
        (
            "no_override",
            no_override_rows.iloc[0],
        )
    )

FINAL_CONFIGS = []
seen = set()

for name, row in candidate_rows:
    config = row_to_config(row)

    key = (
        config["tree_quantile"],
        config["pair_weight"],
        config["round_decimals"],
        config["override_threshold"],
    )

    if key not in seen:
        seen.add(key)
        FINAL_CONFIGS.append(
            (name, config)
        )

FINAL_CONFIGS = FINAL_CONFIGS[:5]

print("최종 제출 후보")
for name, config in FINAL_CONFIGS:
    print(name, config)


최종 제출 후보
cv_best {'tree_quantile': 0.54, 'pair_weight': 0.28, 'round_decimals': 2, 'override_threshold': 0.2, 'cv_mae': 0.14780333333333331}
tree_heavy {'tree_quantile': 0.54, 'pair_weight': 0.25, 'round_decimals': 2, 'override_threshold': 0.15, 'cv_mae': 0.14786333333333335}
pair_heavy {'tree_quantile': 0.54, 'pair_weight': 0.32, 'round_decimals': 2, 'override_threshold': 0.15, 'cv_mae': 0.14787666666666666}
no_override {'tree_quantile': 0.54, 'pair_weight': 0.28, 'round_decimals': 2, 'override_threshold': None, 'cv_mae': 0.14805}


## 8. 전체 데이터 학습 및 제출 파일 생성


In [9]:
if not sample_path.exists():
    submission_template = pd.DataFrame({
        ID_COLUMN: test_raw[ID_COLUMN],
        TARGET: 0.0,
    })
else:
    submission_template = pd.read_csv(
        sample_path
    )

bounds = compute_iqr_bounds(
    raw_input,
    WINSORIZE_COLUMNS,
)

train_w = apply_winsorize(
    raw_input,
    bounds,
)

test_w = apply_winsorize(
    test_input,
    bounds,
)

train_feat = add_features(train_w)
test_feat = add_features(test_w)

test_model_matrices = {}

for model_name in selected_models:
    test_model_matrices[model_name] = (
        fit_extra_trees_matrix(
            train_feat,
            test_feat,
            y,
            MODEL_CONFIGS[model_name],
        )
    )

    print(
        f"전체 데이터 {model_name} 완료"
    )

combined_test_matrix = np.concatenate(
    [
        test_model_matrices[model_name]
        for model_name in selected_models
    ],
    axis=1,
)

test_pair_pred = pair_prediction(
    train_feat,
    test_feat,
    y,
)

test_nearest_dist, test_nearest_target = (
    nearest_neighbor_lookup(
        raw_input,
        test_input,
        y,
    )
)

saved_paths = []

for rank, (name, config) in enumerate(
    FINAL_CONFIGS,
    start=1,
):
    tree_pred = np.quantile(
        combined_test_matrix,
        config["tree_quantile"],
        axis=1,
    )

    blended = (
        (1.0 - config["pair_weight"])
        * tree_pred
        + config["pair_weight"]
        * test_pair_pred
    )

    final_pred = apply_postprocess(
        blended,
        config["round_decimals"],
        config["override_threshold"],
        test_nearest_dist,
        test_nearest_target,
    )

    submission = submission_template.copy()
    submission[TARGET] = final_pred

    filename = (
        f"submit_diverse_et_v11_rank{rank}_{name}"
        f"_models{len(selected_models)}"
        f"_tq{config['tree_quantile']:.3f}"
        f"_w{config['pair_weight']:.2f}"
        f"_th{config['override_threshold']}.csv"
    )

    output_file = OUTPUT_DIR / filename

    submission.to_csv(
        output_file,
        index=False,
    )

    saved_paths.append(
        output_file
    )

    print("\nSaved:", output_file)
    print("Config:", config)
    print(
        "Prediction min/mean/max:",
        float(final_pred.min()),
        float(final_pred.mean()),
        float(final_pred.max()),
    )
    display(submission.head())

print("\n생성 완료")
for path in saved_paths:
    print(path)


전체 데이터 mf1_seed42 완료
전체 데이터 mf2_seed2024 완료
전체 데이터 mf1_boot_seed1024 완료
전체 데이터 sqrt_seed777 완료

Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_diverse_et_v11_rank1_cv_best_models4_tq0.540_w0.28_th0.2.csv
Config: {'tree_quantile': 0.54, 'pair_weight': 0.28, 'round_decimals': 2, 'override_threshold': 0.2, 'cv_mae': 0.14780333333333331}
Prediction min/mean/max: 0.0 0.5099633333333333 1.0


,ID,stress_score
0,TEST_0000,0.62
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.52
4,TEST_0004,0.53



Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_diverse_et_v11_rank2_tree_heavy_models4_tq0.540_w0.25_th0.15.csv
Config: {'tree_quantile': 0.54, 'pair_weight': 0.25, 'round_decimals': 2, 'override_threshold': 0.15, 'cv_mae': 0.14786333333333335}
Prediction min/mean/max: 0.0 0.5105166666666666 1.0


,ID,stress_score
0,TEST_0000,0.62
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.52
4,TEST_0004,0.53



Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_diverse_et_v11_rank3_pair_heavy_models4_tq0.540_w0.32_th0.15.csv
Config: {'tree_quantile': 0.54, 'pair_weight': 0.32, 'round_decimals': 2, 'override_threshold': 0.15, 'cv_mae': 0.14787666666666666}
Prediction min/mean/max: 0.0 0.5087433333333333 1.0


,ID,stress_score
0,TEST_0000,0.62
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.53
4,TEST_0004,0.53



Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_diverse_et_v11_rank4_no_override_models4_tq0.540_w0.28_thNone.csv
Config: {'tree_quantile': 0.54, 'pair_weight': 0.28, 'round_decimals': 2, 'override_threshold': None, 'cv_mae': 0.14805}
Prediction min/mean/max: 0.0 0.5100300000000001 1.0


,ID,stress_score
0,TEST_0000,0.62
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.52
4,TEST_0004,0.53



생성 완료
/Users/bitsaem/Desktop/stress_project/8_6/submit_diverse_et_v11_rank1_cv_best_models4_tq0.540_w0.28_th0.2.csv
/Users/bitsaem/Desktop/stress_project/8_6/submit_diverse_et_v11_rank2_tree_heavy_models4_tq0.540_w0.25_th0.15.csv
/Users/bitsaem/Desktop/stress_project/8_6/submit_diverse_et_v11_rank3_pair_heavy_models4_tq0.540_w0.32_th0.15.csv
/Users/bitsaem/Desktop/stress_project/8_6/submit_diverse_et_v11_rank4_no_override_models4_tq0.540_w0.28_thNone.csv


## 확인할 결과

1. `individual_df`
   - `max_features=2`, `sqrt`, `bootstrap` 등이 단독으로 얼마나 나오는지 확인
2. 예측 상관계수
   - 서로 다른 모델이 충분히 다른 예측을 만드는지 확인
3. `selected_models`
   - 최종 앙상블에 어떤 모델이 포함됐는지 확인
4. `search_df`
   - v10 최고 `0.147603`보다 좋아졌는지 확인
